---
title: "Final Project: Classification"
format: 
  html:
    embed-resources: true
execute:
  echo: true
code-fold: true
author: James Compagno
jupyter: python3
---

# **Classification Models**

## **A. Models That Are Classification Only**

### **Logistic Regression**

* Binary: direct
* Multiclass: uses **OvR (One-vs-Rest)** by default in `sklearn`
* Can be configured to use **multinomial (softmax)**, but class notes emphasize OvR

---

### **LDA — Linear Discriminant Analysis**

* Works for **binary and multiclass**
* **Native multiclass** (no OvR or OvO needed)
* Models each class using Normal distributions with shared variance

---

### **QDA — Quadratic Discriminant Analysis**

* Works for **binary and multiclass**
* **Native multiclass** (no OvR or OvO)
* Similar to LDA but allows different variances per class

---

### **SVC — Support Vector Classifier (linear kernel)**

* Binary: standard SVC
* Multiclass (in `sklearn`): **OvR by default**

```python
SVC(decision_function_shape='ovr')
```

---

### **SVM — Kernel Support Vector Machine (RBF, polynomial, etc.)**

* Same base estimator as SVC in `sklearn`
* Binary: standard SVM
* Multiclass: **OvR by default**
* OvO is possible in theory or other implementations

# **GSB 544 – Fall 2025 – Classification**

### *Predict a person's political affiliation from their responses to survey questions.*

---

## **The Data**

This dataset comes from a survey conducted in **March 2018** by Survey Sampling International, commissioned by **Cards Against Humanity**.

The survey includes demographic questions and opinion-based questions relating to legality, morality, sexuality, religion, and political attitudes.

The **bolded question is the target variable** for prediction.

---

## **Survey Questions Included in the Dataset (with column names)**

* **Q1** — *What gender do you identify with?*
* **Q2** — *Age*
* **political_affiliation** — **In politics today, do you consider yourself a Democrat, a Republican, or Independent?** *(Target variable)*
* **Q4** — *Would you say you are liberal, conservative, or moderate?*
* **Q5** — *What is your highest level of education?*
* **Q6** — *What is your race?*
* **Q7** — *Do you believe that prostitution should be against the law?*
* **Q8** — *Do you believe that it should be against the law to smoke marijuana for recreation?*
* **Q9** — *Do you believe the sale of human organs for transplants should be against the law?*
* **Q10** — *Would you consider yourself a religious person?*
* **Q11** — *Would you consider yourself pro-life or pro-choice?*
* **Q12** — *Do you generally disapprove of people who pursue casual sex, hookups, or one-night stands?*
* **Q13** — *Based on your general impressions, would you say that people who smoke marijuana tend to have more casual sex than people who abstain?*
* **Q14** — *If abortion became strictly banned, do you think women would become more willing to have casual sex, less willing, or behave no differently?*
* **Q15** — *Do you agree/disagree: “A woman should have the right to do what she wants with her own body.”*
* **Q16** — *Do you agree/disagree: “Abortion is morally wrong.”*
* **Q17** — *Do you agree/disagree: “Sex without love is okay.”*
* **Q18** — *Do you believe an elected official who has committed sexual misconduct in their personal life can still behave ethically in office?*

In [26]:
import numpy as np
import pandas as pd
import plotnine as p9
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import (
    LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
)
from sklearn.svm import SVC
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score,
    precision_recall_fscore_support, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc, precision_score,
    recall_score, f1_score
)
from sklearn.model_selection import (
    GridSearchCV, cross_val_score, train_test_split, cross_val_predict
)
from sklearn.pipeline import Pipeline

In [27]:
# Read the data
CAH_train = pd.read_csv("/Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-classification/CAH-201803-train.csv")

# Load data and prepare features/target
X = CAH_train.drop(["id_num", "political_affiliation"], axis=1)
y = CAH_train["political_affiliation"]

# Model Library 
model_library = {}
records = []

# Read test data
CAH_test = pd.read_csv("/Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-classification/CAH-201803-test.csv")
X_test = CAH_test.drop("id_num", axis=1)
test_ids = CAH_test["id_num"]

In [28]:
# Define feature lists
numerical_features = ["Q2", "Q15", "Q16", "Q17"] 

categorical_features = ["Q1", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9", 
                        "Q10", "Q11", "Q12", "Q13", "Q14", "Q18"]

# Column Transformer 
ct = ColumnTransformer(
    [
        ("standardize", 
         StandardScaler(), 
         numerical_features),
        ("encode",
         OneHotEncoder(drop='first', sparse_output=False),
         categorical_features)
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# Q1: KNN

In [29]:
def knn_gridsearch(model_name, features=None, k_range=None, weights=None, p_values=None):
    """
    Uses GridSearchCV to find the best hyperparameters for KNN (multi-class)
    model_name - will be stored in model_library
    k_range - list of k values to test 
    weights - list of weight options ['uniform', 'distance']
    p_values - list of p values for Minkowski distance [1, 2]
    features - list or None 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Set defaults if not provided
    if k_range is None:
        k_range = [5]
    if weights is None:
        weights = ['uniform']
    if p_values is None:
        p_values = [2]
    
    # Pipeline with KNN Classifier
    pipe = Pipeline([
        ("preprocess", ct),
        ("knn", KNeighborsClassifier())
    ])
    
    # Parameter grid - now includes multiple hyperparameters
    param_grid = {
        'knn__n_neighbors': k_range,
        'knn__weights': weights,
        'knn__p': p_values
    }
    
    # GridSearchCV - using ACCURACY as primary metric
    grid_search = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring='accuracy',
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_subset, y)
    
    # Store best model
    best_model = grid_search.best_estimator_
    model_library[model_name] = best_model
    
    # Best parameters
    best_k = grid_search.best_params_['knn__n_neighbors']
    best_weights = grid_search.best_params_['knn__weights']
    best_p = grid_search.best_params_['knn__p']
    
    # Cross-validated predictions
    y_pred_cv = cross_val_predict(best_model, X_subset, y, cv=5)
    y_proba_cv = cross_val_predict(best_model, X_subset, y, cv=5, method='predict_proba')
    
    # Metrics on CV predictions (multi-class)
    conf_matrix = confusion_matrix(y, y_pred_cv)
    cv_accuracy = accuracy_score(y, y_pred_cv)
    
    # Multi-class ROC AUC (One-vs-Rest)
    cv_roc_auc = roc_auc_score(y, y_proba_cv, multi_class='ovr', average='weighted')
    
    # Precision, Recall, F1 (weighted average across classes)
    precision = precision_score(y, y_pred_cv, average='weighted', zero_division=0)
    recall = recall_score(y, y_pred_cv, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred_cv, average='weighted', zero_division=0)
    
    # Store results
    records.append({
        "Model": model_name,
        "Classification Type": "KNN",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Hyperparameter 1 Name": "n_neighbors", 
        "Hyperparameter 1 Value": best_k,
        "Hyperparameter 2 Name": "weights", 
        "Hyperparameter 2 Value": best_weights,
        "Hyperparameter 3 Name": "p",
        "Hyperparameter 3 Value": best_p,
        "Range Tested (k)": str(k_range),
        "Range Tested (weights)": str(weights),
        "Range Tested (p)": str(p_values),
        "CV Accuracy": cv_accuracy,
        "ROC AUC (weighted)": cv_roc_auc,
        "Precision (weighted)": precision,
        "Recall (weighted)": recall,
        "F1 (weighted)": f1,
        "Confusion Matrix": conf_matrix,
    })
    
    # Print results
    print(f"Best Parameters:")
    print(f"  n_neighbors (K): {best_k}")
    print(f"  weights: {best_weights}")
    print(f"  p (distance metric): {best_p} ({'manhattan' if best_p == 1 else 'euclidean' if best_p == 2 else 'minkowski'})")
    print(f"\nCV Accuracy: {cv_accuracy:.4f}")
    print(f"ROC AUC (weighted): {cv_roc_auc:.4f}")
    print(f"\nConfusion Matrix (CV):")
    print(conf_matrix)
    print(f"\nClassification Report:")
    print(classification_report(y, y_pred_cv))

    return

In [30]:
knn_gridsearch(
    "KNN_broad", 
    features=None, 
    k_range=list(range(1, 51, 2)),  
    weights=['uniform', 'distance'],
    p_values=[1, 2]
)

Best Parameters:
  n_neighbors (K): 13
  weights: distance
  p (distance metric): 1 (manhattan)

CV Accuracy: 0.5740
ROC AUC (weighted): 0.7487

Confusion Matrix (CV):
[[34 16  9]
 [18 30  8]
 [ 8 13 33]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.57      0.58      0.57        59
 Independent       0.51      0.54      0.52        56
  Republican       0.66      0.61      0.63        54

    accuracy                           0.57       169
   macro avg       0.58      0.57      0.58       169
weighted avg       0.58      0.57      0.58       169



In [31]:
knn_gridsearch(
    "KNN_fine", 
    features=None, 
    k_range=list(range(7, 21)),  # 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20
    weights=['uniform', 'distance'],
    p_values=[1]
)

Best Parameters:
  n_neighbors (K): 16
  weights: distance
  p (distance metric): 1 (manhattan)

CV Accuracy: 0.5799
ROC AUC (weighted): 0.7503

Confusion Matrix (CV):
[[34 17  8]
 [18 30  8]
 [ 6 14 34]]

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.59      0.58      0.58        59
 Independent       0.49      0.54      0.51        56
  Republican       0.68      0.63      0.65        54

    accuracy                           0.58       169
   macro avg       0.59      0.58      0.58       169
weighted avg       0.58      0.58      0.58       169



In [32]:
dfKNN = pd.DataFrame(records)
dfKNN[dfKNN['Classification Type'] == 'KNN'].sort_values('CV Accuracy', ascending=False)

,Model,Classification Type,Variables Used,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Range Tested (k),Range Tested (weights),Range Tested (p),CV Accuracy,ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Confusion Matrix
1,KNN_fine,KNN,All,n_neighbors,16,weights,distance,p,1,"[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, ...","['uniform', 'distance']",[1],0.579882,0.750268,0.584895,0.579882,0.581753,"[[34, 17, 8], [18, 30, 8], [6, 14, 34]]"
0,KNN_broad,KNN,All,n_neighbors,13,weights,distance,p,1,"[1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25...","['uniform', 'distance']","[1, 2]",0.573964,0.748652,0.577207,0.573964,0.575153,"[[34, 16, 9], [18, 30, 8], [8, 13, 33]]"
